In [1]:
import os

In [2]:
%pwd

'p:\\OM\\Projects\\End-to-End-ML-\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'p:\\OM\\Projects\\End-to-End-ML-'

In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen = True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    TARGET_COLUMN: str

In [10]:
from src.wine.constants import *
from src.wine.utils.common import read_yaml, create_directories, save_json

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH,
        schema_file_path = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema =  self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path = config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            TARGET_COLUMN = schema.name
           
        )

        return model_evaluation_config

In [8]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import joblib

In [11]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self,actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    
    def save_results(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        x_test = test_data.drop([self.config.TARGET_COLUMN], axis=1)
        y_test = test_data[[self.config.TARGET_COLUMN]]
        
        y_pred = model.predict(x_test)

        (rmse, mae, r2) = self.eval_metrics(y_test, y_pred)
        
        # Saving metrics as local
        scores = {"rmse": rmse, "mae": mae, "r2": r2}
        save_json(path=Path(self.config.metric_file_name), data=scores)

In [12]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.save_results()
except Exception as e:
    raise e

[ 2026-06-22 16:06:28,174 : INFO : common : yaml file: config\config.yaml loaded successfully ]
[ 2026-06-22 16:06:28,180 : INFO : common : yaml file: params.yaml loaded successfully ]
[ 2026-06-22 16:06:28,184 : INFO : common : yaml file: schema.yaml loaded successfully ]
[ 2026-06-22 16:06:28,186 : INFO : common : created directory at: artifacts ]
[ 2026-06-22 16:06:28,188 : INFO : common : created directory at: artifacts/model_evaluation ]
[ 2026-06-22 16:06:28,386 : INFO : common : json file saved at: artifacts\model_evaluation\metrics.json ]
